# Lab 05: RAG Evaluation & Optimization

## 🎯 Objectives

Systematically compare RAG configurations using evaluators:
1. Establish baseline with default RAG configuration
2. Run Retrieval, Groundedness, Relevance, Response Completeness evaluators
3. Compare configurations (chunk size, top-k, retrieval mode)
4. Identify optimal configuration through data

## 💡 RAG Failure Decomposition

RAG failures can occur at any stage:
```
Query Understanding → Retrieval → Ranking → Context Construction → Generation → Citation
```

Each stage needs different evaluators:
- **Retrieval quality**: Did we find the right documents?
- **Groundedness (Precision)**: Is the answer supported by retrieved context?
- **Completeness (Recall)**: Does the answer cover all expected information?
- **Relevance**: Does it actually answer the question?

## ⏱️ Duration: 60-75 minutes

In [ ]:
import os
import json
import time
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

env_path = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / '.env.template').is_file()) / '.env'
load_dotenv(env_path)

project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
tenant_id = os.environ.get("TENANT_ID")
model_deployment = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

print(f"📍 Project: {project_endpoint[:50]}...")
print(f"🤖 Model: {model_deployment}")

In [ ]:
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient

credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("✅ Clients initialized")

## Step 1: Define RAG Configurations to Compare

We'll test different parameter combinations to find the optimal setup.

In [ ]:
# RAG configurations to compare
rag_configs = [
    {
        "name": "baseline",
        "description": "Default configuration",
        "chunk_size": 512,
        "chunk_overlap": 50,
        "top_k": 3,
        "retrieval_mode": "vector",
        "system_prompt": "You are a helpful assistant. Answer based only on the provided context."
    },
    {
        "name": "larger_chunks",
        "description": "Larger chunk size for more context",
        "chunk_size": 1024,
        "chunk_overlap": 100,
        "top_k": 3,
        "retrieval_mode": "vector",
        "system_prompt": "You are a helpful assistant. Answer based only on the provided context."
    },
    {
        "name": "more_results",
        "description": "More documents retrieved (top-5)",
        "chunk_size": 512,
        "chunk_overlap": 50,
        "top_k": 5,
        "retrieval_mode": "vector",
        "system_prompt": "You are a helpful assistant. Answer based only on the provided context."
    },
    {
        "name": "hybrid_search",
        "description": "Hybrid (vector + keyword) retrieval",
        "chunk_size": 512,
        "chunk_overlap": 50,
        "top_k": 3,
        "retrieval_mode": "hybrid",
        "system_prompt": "You are a helpful assistant. Answer based only on the provided context."
    },
    {
        "name": "detailed_prompt",
        "description": "More detailed system message with instructions",
        "chunk_size": 512,
        "chunk_overlap": 50,
        "top_k": 3,
        "retrieval_mode": "vector",
        "system_prompt": """You are a Contoso policy expert. Answer employee questions using ONLY the provided context.
Rules:
- Cite the specific policy section when possible
- If the context doesn't contain the answer, say 'I don't have that information in the available policies'
- Be concise but complete
- Include specific numbers (days, amounts, deadlines) when available"""
    }
]

print(f"📊 {len(rag_configs)} configurations to compare:")
for cfg in rag_configs:
    print(f"   • {cfg['name']}: {cfg['description']}")

## Step 2: Define RAG Test Cases

These test cases have both context (retrieved documents) and ground truth for full RAG evaluation.

In [ ]:
# RAG-specific test cases with context and ground truth
rag_test_cases = [
    {
        "query": "How many days of annual leave do full-time employees get?",
        "context": "HR Policy 3.1: Annual Leave. Full-time employees are entitled to 20 working days of annual leave per calendar year. Leave accrues at 1.67 days per month.",
        "ground_truth": "Full-time employees receive 20 days of annual leave per year, accruing at 1.67 days per month.",
        "category": "direct_answer",
        "doc_type": "hr_policy"
    },
    {
        "query": "What is the expense report submission deadline?",
        "context": "Finance Policy 7.2: Expense Reporting. All business expenses must be reported within 30 days of incurrence via the Finance Portal.",
        "ground_truth": "Expense reports must be submitted within 30 days of the expense via the Finance Portal.",
        "category": "direct_answer",
        "doc_type": "finance_policy"
    },
    {
        "query": "What happens if I need emergency leave but haven't completed 6 months?",
        "context": "HR Policy 3.3: Emergency Leave. Up to 5 days paid emergency leave available to all employees regardless of tenure for immediate family emergencies. HR Policy 3.1: Annual leave requires 6 months tenure for standard requests.",
        "ground_truth": "Emergency leave of up to 5 days is available regardless of tenure for family emergencies. Standard annual leave requires 6 months, but emergency leave does not.",
        "category": "multi_doc_reasoning",
        "doc_type": "hr_policy"
    },
    {
        "query": "Can part-time employees get parental leave?",
        "context": "HR Policy 3.5: Parental Leave. Primary caregivers: 16 weeks fully paid. Secondary caregivers: 4 weeks fully paid. Applies to full-time employees only. Part-time employees may be eligible for statutory parental leave per local regulations.",
        "ground_truth": "The company's enhanced parental leave (16/4 weeks) applies to full-time employees only. Part-time employees may be eligible for statutory parental leave as required by local regulations.",
        "category": "nuanced_answer",
        "doc_type": "hr_policy"
    },
    {
        "query": "What's the budget for professional development?",
        "context": "HR Policy 4.3: Professional Development. Each employee receives $2,000 per calendar year for approved professional development activities including courses, conferences, and certifications. Unused budget does not roll over.",
        "ground_truth": "Each employee gets $2,000 per year for professional development (courses, conferences, certifications). The budget does not roll over to the next year.",
        "category": "direct_answer",
        "doc_type": "hr_policy"
    },
]

print(f"📝 {len(rag_test_cases)} RAG test cases loaded")
for tc in rag_test_cases:
    print(f"   [{tc['category']:20}] {tc['query'][:60]}")

## Step 3: Simulate RAG Responses per Configuration

For each configuration, generate responses and score them.

In [ ]:
def generate_rag_response(query: str, context: str, system_prompt: str) -> str:
    """Generate a RAG response using the specified system prompt."""
    messages = [
        {"role": "system", "content": f"{system_prompt}\n\nContext:\n{context}"},
        {"role": "user", "content": query}
    ]
    response = openai_client.chat.completions.create(
        model=model_deployment,
        messages=messages,
        temperature=0,
        max_tokens=500
    )
    return response.choices[0].message.content

# Generate responses for each config
config_responses = {}

for config in rag_configs:
    print(f"\n🔄 Generating responses for: {config['name']}")
    responses = []
    for tc in rag_test_cases:
        response = generate_rag_response(tc['query'], tc['context'], config['system_prompt'])
        responses.append({
            "query": tc['query'],
            "response": response,
            "context": tc['context'],
            "ground_truth": tc['ground_truth'],
            "category": tc['category'],
        })
    config_responses[config['name']] = responses
    print(f"   ✅ {len(responses)} responses generated")

print(f"\n✅ All configurations processed")

## Step 4: Score Responses with RAG Evaluators

Use LLM-as-judge to evaluate groundedness and relevance per configuration.

In [ ]:
# Simplified scoring using LLM judge for demonstration
GROUNDEDNESS_PROMPT = """Rate how well the response is grounded in (supported by) the provided context.
Score 1-5:
5: Fully grounded, every claim supported
4: Mostly grounded, minor unsupported details
3: Partially grounded, some claims lack support
2: Mostly ungrounded, significant fabrication
1: Completely ungrounded, contradicts context

Context: {context}
Response: {response}

Output JSON only: {{"score": <1-5>, "reasoning": "<brief>"}}"""

COMPLETENESS_PROMPT = """Rate how completely the response covers the expected information.
Score 1-5:
5: Covers all key points from ground truth
4: Covers most key points, minor omissions
3: Covers main point but misses important details
2: Significant information missing
1: Barely addresses the question

Expected (ground truth): {ground_truth}
Actual response: {response}

Output JSON only: {{"score": <1-5>, "reasoning": "<brief>"}}"""

def score_response(response: str, context: str, ground_truth: str) -> dict:
    """Score a single response on groundedness and completeness."""
    scores = {}
    
    # Groundedness
    g_result = openai_client.chat.completions.create(
        model=model_deployment,
        messages=[{"role": "user", "content": GROUNDEDNESS_PROMPT.format(context=context, response=response)}],
        temperature=0, response_format={"type": "json_object"}
    )
    scores["groundedness"] = json.loads(g_result.choices[0].message.content)["score"]
    
    # Completeness
    c_result = openai_client.chat.completions.create(
        model=model_deployment,
        messages=[{"role": "user", "content": COMPLETENESS_PROMPT.format(ground_truth=ground_truth, response=response)}],
        temperature=0, response_format={"type": "json_object"}
    )
    scores["completeness"] = json.loads(c_result.choices[0].message.content)["score"]
    
    return scores

# Score all configurations
all_scores = {}
for config_name, responses in config_responses.items():
    print(f"\n📊 Scoring: {config_name}")
    config_scores = []
    for r in responses:
        scores = score_response(r['response'], r['context'], r['ground_truth'])
        scores['query'] = r['query'][:40]
        scores['category'] = r['category']
        config_scores.append(scores)
    all_scores[config_name] = config_scores
    
    avg_g = sum(s['groundedness'] for s in config_scores) / len(config_scores)
    avg_c = sum(s['completeness'] for s in config_scores) / len(config_scores)
    print(f"   Avg Groundedness: {avg_g:.2f}/5, Avg Completeness: {avg_c:.2f}/5")

## Step 5: Compare Configurations

In [ ]:
# Build comparison table
comparison = []
for config_name, scores in all_scores.items():
    avg_g = sum(s['groundedness'] for s in scores) / len(scores)
    avg_c = sum(s['completeness'] for s in scores) / len(scores)
    combined = (avg_g + avg_c) / 2
    comparison.append({
        "Configuration": config_name,
        "Groundedness": f"{avg_g:.2f}",
        "Completeness": f"{avg_c:.2f}",
        "Combined": f"{combined:.2f}",
    })

df_comparison = pd.DataFrame(comparison)
df_comparison = df_comparison.sort_values('Combined', ascending=False)

print("📊 RAG Configuration Comparison")
print("=" * 60)
display(df_comparison)

best = df_comparison.iloc[0]
print(f"\n🏆 Best Configuration: {best['Configuration']} (Combined: {best['Combined']})")

In [ ]:
# Save RAG optimization report
report = {
    "title": "RAG Optimization Report",
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "configurations_tested": len(rag_configs),
    "test_cases": len(rag_test_cases),
    "results": comparison,
    "recommendation": best['Configuration'],
    "recommendation_scores": {"groundedness": best['Groundedness'], "completeness": best['Completeness']}
}

with open("./rag-optimization-report.json", "w") as f:
    json.dump(report, f, indent=2)

print("✅ RAG Optimization Report saved to ./rag-optimization-report.json")

## ✅ Lab 05 Complete

### What You Created
- Compared 5 RAG configurations systematically
- Scored each on Groundedness + Completeness
- Identified optimal configuration through data
- **RAG Optimization Report** with recommendations

### Key Insight
- Groundedness = Precision (no hallucination)
- Completeness = Recall (all info covered)
- Often in tension — optimize for your use case priority

### Next Steps
- **Lab 06**: Add agent-specific tool trajectory evaluation
- Apply the optimal config to your production agent